# RESUME: WaveFusion-Net Fixed-Fusion (no learned gate) -- RealBlur-J, continuing from epoch 50

**Why this exists.** The from-scratch run `run_nogate_realblurj_20260923_164622` was interrupted after
epoch 50 of 70. Windows logged a sleep request at 2026-09-24 03:42:48 and the machine stayed asleep
until the lid was opened at 08:17 (the two earlier starts that day also died right after sleep requests).
The only thing on disk was the weights-only checkpoint from epoch 50: VAL PSNR 30.0121 dB, SSIM 0.8853.

**What this notebook does.** Loads those epoch-50 weights and trains epochs 51-70 (about 3.5 hours).
- Fresh AdamW state: the checkpoint is weights-only, so optimizer moments are not available.
- Cosine LR schedule restored to its exact epoch-50 position (LR for epoch 51 = 3.77e-5, the same value
  the full-model log shows for epoch 51).
- Everything else identical to the from-scratch notebook: seed-42 split, five-component loss, batch 1,
  grad-accum 2, patch 192, AMP, eval at epochs 60 and 70, final TEST once on the best-VAL weights.

**If you ever report this result, say so:** "fixed-fusion, resumed at epoch 50 with re-initialised
optimiser state". It is not a bit-for-bit uninterrupted run.

**Checks that run BEFORE any training (each one stops the notebook with a clear error if it fails):**
1. train/val/test SHA256 hashes must equal the interrupted run's split proof.
2. The cosine schedule must land on the expected epoch-50 learning rate.
3. The loaded weights must reproduce the stored epoch-50 VAL PSNR (30.0121 dB) on the real VAL set.

**Safeguards added because of what happened:**
- `ablation_nogate_realblurj_history.json` is rewritten after every epoch and every eval (curves and
  numbers can no longer be lost to a mid-run stop; TEST numbers are written into it at the end too).
- The training cell asks Windows not to idle-sleep while it runs (a per-process request that ends with the
  kernel; it changes no Windows setting).

## Before you press Run All (this is what killed all three earlier runs)
1. Settings > System > Power & battery > Screen, sleep & hibernate > "When plugged in, put my device to
   sleep after" = **Never**.
2. Control Panel > Power Options > "Choose what closing the lid does" > Plugged in = **Do nothing**.
3. Keep the laptop plugged in. Do not press Sleep / the power button, and do not use Start > Sleep.

Outputs go to `ABLATION\runs\run_nogate_realblurj_RESUME_<timestamp>\`. Your epoch-50 checkpoint in the old
run folder is only read, never modified.

In [ ]:
get_ipython().getoutput("pip install torchinfo")

In [ ]:
import gc
import hashlib
import json
import os
import random
import time
from glob import glob

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from tqdm import tqdm

try:
    from torchinfo import summary as torchinfo_summary
except ImportError:
    torchinfo_summary = None

## Data root resolution (copied verbatim from the real RealBlur-J source) + reproducibility utilities

In [ ]:
def _has_realblur_structure(realblur_root):
    '''Returns True if `realblur_root` looks like .../RealBlur with train/test folders.'''
    train_blur = os.path.join(realblur_root, "train", "blur")
    train_gt = os.path.join(realblur_root, "train", "gt")
    test_blur = os.path.join(realblur_root, "test", "blur")
    test_gt = os.path.join(realblur_root, "test", "gt")
    if not (os.path.isdir(train_blur) and os.path.isdir(train_gt) and os.path.isdir(test_blur) and os.path.isdir(test_gt)):
        return False
    for ext in ("png", "jpg", "jpeg", "PNG", "JPG", "JPEG"):
        if glob(os.path.join(train_blur, f"*.{ext}")):
            return True
    return False


def resolve_realblur_root(preferred):
    '''Resolve the RealBlur-J root path in Kaggle.'''
    if preferred and _has_realblur_structure(preferred):
        return preferred
    search_base = "/kaggle/input"
    if not os.path.isdir(search_base):
        return preferred  # non-Kaggle environment
    patterns = [
        os.path.join(search_base, "**", "RealBlur", "train", "blur", "*.png"),
        os.path.join(search_base, "**", "RealBlur", "train", "blur", "*.jpg"),
        os.path.join(search_base, "**", "RealBlur", "train", "blur", "*.jpeg"),
    ]
    for pat in patterns:
        matches = glob(pat, recursive=True)
        if matches:
            m = matches[0]
            realblur_root = os.path.dirname(os.path.dirname(os.path.dirname(m)))
            if _has_realblur_structure(realblur_root):
                return realblur_root
    return preferred


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sha256_of_list(lines):
    h = hashlib.sha256()
    for line in lines:
        h.update((line + "\n").encode("utf-8"))
    return h.hexdigest()


def export_split_proof(save_dir, train_relpaths, val_relpaths, test_relpaths, config):
    os.makedirs(save_dir, exist_ok=True)
    train_sorted = sorted(train_relpaths)
    val_sorted = sorted(val_relpaths)
    test_sorted = sorted(test_relpaths)
    proof = {
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "ablation": "fixed_fusion_no_learned_gate",
        "dataset": "realblurj",
        "val_ratio": config.get("val_ratio"),
        "split_seed": config.get("split_seed"),
        "counts": {"train": len(train_sorted), "val": len(val_sorted), "test": len(test_sorted)},
        "sha256": {
            "train": sha256_of_list(train_sorted),
            "val": sha256_of_list(val_sorted),
            "test": sha256_of_list(test_sorted),
        },
        "notes": {"protocol": "best checkpoint selected on VAL-from-TRAIN; same seed/split as full-model and spatial-only RealBlur-J runs"},
    }
    out_path = os.path.join(save_dir, "split_proof_ablation_nogate_realblurj.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(proof, f, indent=2)
    return out_path

## Model: full dual-branch architecture with CrossBranchFusion's learned gate replaced by a fixed 0.5/0.5 blend (byte-identical to the GoPro fixed-fusion ablation)

In [ ]:
class DWT(nn.Module):
    '''Discrete Wavelet Transform using Haar wavelets'''

    def __init__(self):
        super().__init__()
        self.register_buffer("ll", torch.tensor([[0.5, 0.5], [0.5, 0.5]]).view(1, 1, 2, 2))
        self.register_buffer("lh", torch.tensor([[0.5, 0.5], [-0.5, -0.5]]).view(1, 1, 2, 2))
        self.register_buffer("hl", torch.tensor([[0.5, -0.5], [0.5, -0.5]]).view(1, 1, 2, 2))
        self.register_buffer("hh", torch.tensor([[0.5, -0.5], [-0.5, 0.5]]).view(1, 1, 2, 2))

    def forward(self, x):
        B, C, H, W = x.shape
        if H % 2 != 0:
            x = F.pad(x, (0, 0, 0, 1))
        if W % 2 != 0:
            x = F.pad(x, (0, 1, 0, 0))
        x = x.contiguous().view(B * C, 1, x.shape[2], x.shape[3])
        ll = F.conv2d(x, self.ll, stride=2)
        lh = F.conv2d(x, self.lh, stride=2)
        hl = F.conv2d(x, self.hl, stride=2)
        hh = F.conv2d(x, self.hh, stride=2)
        ll = ll.view(B, C, ll.shape[2], ll.shape[3])
        lh = lh.view(B, C, lh.shape[2], lh.shape[3])
        hl = hl.view(B, C, hl.shape[2], hl.shape[3])
        hh = hh.view(B, C, hh.shape[2], hh.shape[3])
        return ll, lh, hl, hh


class IDWT(nn.Module):
    '''Inverse Discrete Wavelet Transform (unused in forward -- kept for
    architectural fidelity with the full model).'''

    def __init__(self):
        super().__init__()
        self.register_buffer("ll", torch.tensor([[0.5, 0.5], [0.5, 0.5]]).view(1, 1, 2, 2))
        self.register_buffer("lh", torch.tensor([[0.5, 0.5], [-0.5, -0.5]]).view(1, 1, 2, 2))
        self.register_buffer("hl", torch.tensor([[0.5, -0.5], [0.5, -0.5]]).view(1, 1, 2, 2))
        self.register_buffer("hh", torch.tensor([[0.5, -0.5], [-0.5, 0.5]]).view(1, 1, 2, 2))

    def forward(self, ll, lh, hl, hh):
        B, C, H, W = ll.shape
        ll = ll.view(B * C, 1, H, W)
        lh = lh.view(B * C, 1, H, W)
        hl = hl.view(B * C, 1, H, W)
        hh = hh.view(B * C, 1, H, W)
        ll = F.conv_transpose2d(ll, self.ll, stride=2)
        lh = F.conv_transpose2d(lh, self.lh, stride=2)
        hl = F.conv_transpose2d(hl, self.hl, stride=2)
        hh = F.conv_transpose2d(hh, self.hh, stride=2)
        out = (ll + lh + hl + hh).view(B, C, ll.shape[2], ll.shape[3])
        return out


class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class StripAttention(nn.Module):
    def __init__(self, channels, strip_size=7):
        super().__init__()
        self.h_conv = nn.Conv2d(channels, channels, (1, strip_size), padding=(0, strip_size // 2), groups=channels)
        self.v_conv = nn.Conv2d(channels, channels, (strip_size, 1), padding=(strip_size // 2, 0), groups=channels)
        self.proj = nn.Conv2d(channels * 2, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h_out = self.h_conv(x)[:, :, :H, :W]
        v_out = self.v_conv(x)[:, :, :H, :W]
        h_attn = torch.sigmoid(h_out)
        v_attn = torch.sigmoid(v_out)
        combined = torch.cat([x * h_attn, x * v_attn], dim=1)
        return self.proj(combined)


class SCA(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        return x * self.fc(self.gap(x))


class NAFBlock(nn.Module):
    def __init__(self, channels, dw_expand=2):
        super().__init__()
        dw_channels = channels * dw_expand
        self.conv1 = nn.Conv2d(channels, dw_channels, 1)
        self.conv2 = nn.Conv2d(dw_channels, dw_channels, 3, padding=1, groups=dw_channels)
        self.conv3 = nn.Conv2d(dw_channels // 2, channels, 1)
        self.sca = SCA(dw_channels // 2)
        self.sg = SimpleGate()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        residual = x
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg(x)
        x = self.sca(x)
        x = self.conv3(x)
        return x + residual


class WaveletBlock(nn.Module):
    '''Process wavelet coefficients with attention to high-frequency details'''

    def __init__(self, channels):
        super().__init__()
        self.ll_conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )
        self.hf_conv = nn.Sequential(
            nn.Conv2d(channels * 3, channels * 3, 3, padding=1, groups=3),
            nn.GELU(),
            nn.Conv2d(channels * 3, channels * 3, 3, padding=1, groups=3),
        )
        self.hf_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels * 3, channels * 3, 1),
            nn.Sigmoid(),
        )

    def forward(self, ll, lh, hl, hh):
        ll_out = ll + self.ll_conv(ll)
        hf = torch.cat([lh, hl, hh], dim=1)
        hf_feat = self.hf_conv(hf)
        hf_attn = self.hf_attn(hf_feat)
        hf_out = hf + hf_feat * hf_attn
        lh_out, hl_out, hh_out = hf_out.chunk(3, dim=1)
        return ll_out, lh_out, hl_out, hh_out


class FixedFusion(nn.Module):
    '''CrossBranchFusion with the learned sigmoid gate removed. Same
    spatial_proj / wavelet_proj / out convs as the full model -- only the
    `Conv2d(channels*2, channels, 1) + Sigmoid` gate submodule is gone,
    replaced by a fixed 0.5/0.5 blend of the two projected features.'''

    def __init__(self, channels):
        super().__init__()
        self.spatial_proj = nn.Conv2d(channels, channels, 1)
        self.wavelet_proj = nn.Conv2d(channels, channels, 1)
        self.out = nn.Conv2d(channels, channels, 1)

    def forward(self, spatial_feat, wavelet_feat):
        if spatial_feat.shape[2:] != wavelet_feat.shape[2:]:
            wavelet_feat = F.interpolate(wavelet_feat, size=spatial_feat.shape[2:], mode="bilinear", align_corners=False)
        s = self.spatial_proj(spatial_feat)
        w = self.wavelet_proj(wavelet_feat)
        fused = 0.5 * s + 0.5 * w
        return self.out(fused)


class WaveFusionNetFixedFusion(nn.Module):
    '''Full dual-branch WaveFusion-Net (spatial NAFBlock branch + wavelet
    WaveletBlock branch, both present) with CrossBranchFusion's learned gate
    replaced by FixedFusion.'''

    def __init__(self, in_channels=3, out_channels=3, base_channels=48, num_blocks=(4, 6, 6, 4)):
        super().__init__()
        self.dwt = DWT()
        self.idwt = IDWT()

        self.intro = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        self.enc1 = nn.Sequential(*[NAFBlock(base_channels) for _ in range(num_blocks[0])])
        self.down1 = nn.Conv2d(base_channels, base_channels * 2, 2, stride=2)

        self.enc2 = nn.Sequential(*[NAFBlock(base_channels * 2) for _ in range(num_blocks[1])])
        self.down2 = nn.Conv2d(base_channels * 2, base_channels * 4, 2, stride=2)

        self.enc3 = nn.Sequential(*[NAFBlock(base_channels * 4) for _ in range(num_blocks[2])])
        self.down3 = nn.Conv2d(base_channels * 4, base_channels * 8, 2, stride=2)

        self.wav_intro = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        self.wav_block1 = WaveletBlock(base_channels)
        self.wav_proj1 = nn.Conv2d(base_channels, base_channels * 2, 1)
        self.wav_block2 = WaveletBlock(base_channels * 2)
        self.wav_proj2 = nn.Conv2d(base_channels * 2, base_channels * 4, 1)
        self.wav_block3 = WaveletBlock(base_channels * 4)

        self.fusion1 = FixedFusion(base_channels * 2)
        self.fusion2 = FixedFusion(base_channels * 4)

        self.bottleneck = nn.Sequential(
            NAFBlock(base_channels * 8),
            StripAttention(base_channels * 8),
            NAFBlock(base_channels * 8),
            NAFBlock(base_channels * 8),
        )

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.dec3 = nn.Sequential(*[NAFBlock(base_channels * 4) for _ in range(num_blocks[2])])

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec2 = nn.Sequential(*[NAFBlock(base_channels * 2) for _ in range(num_blocks[1])])

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = nn.Sequential(*[NAFBlock(base_channels) for _ in range(num_blocks[0])])

        self.refine = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
        )

        self.outro = nn.Conv2d(base_channels, out_channels, 3, padding=1)

    def forward(self, x):
        B, C, H, W = x.shape
        pad_h = (8 - H % 8) % 8
        pad_w = (8 - W % 8) % 8
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")

        f0 = self.intro(x)
        f1 = self.enc1(f0)
        f1_down = self.down1(f1)

        f2 = self.enc2(f1_down)
        f2_down = self.down2(f2)

        f3 = self.enc3(f2_down)
        f3_down = self.down3(f3)

        w0 = self.wav_intro(x)

        ll1, lh1, hl1, hh1 = self.dwt(w0)
        ll1, lh1, hl1, hh1 = self.wav_block1(ll1, lh1, hl1, hh1)
        w1 = self.wav_proj1(ll1)

        ll2, lh2, hl2, hh2 = self.dwt(w1)
        ll2, lh2, hl2, hh2 = self.wav_block2(ll2, lh2, hl2, hh2)
        w2 = self.wav_proj2(ll2)

        ll3, lh3, hl3, hh3 = self.dwt(w2)
        ll3, lh3, hl3, hh3 = self.wav_block3(ll3, lh3, hl3, hh3)

        f2_fused = self.fusion1(f2, w1)
        f3_fused = self.fusion2(f3, w2)

        bottleneck_out = self.bottleneck(f3_down)

        d3 = self.up3(bottleneck_out) + f3_fused
        d3 = self.dec3(d3)

        d2 = self.up2(d3) + f2_fused
        d2 = self.dec2(d2)

        d1 = self.up1(d2) + f1
        d1 = self.dec1(d1)

        out = self.refine(d1)
        out = out + f0
        out = self.outro(out)
        out = out + x

        if pad_h > 0 or pad_w > 0:
            out = out[:, :, :H, :W]
        return out

## Loss functions (unchanged) and dataset (RealBlur-J, copied verbatim from the real source)

In [ ]:
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        self.slice1 = nn.Sequential(*list(vgg.children())[:4])
        self.slice2 = nn.Sequential(*list(vgg.children())[4:9])
        self.slice3 = nn.Sequential(*list(vgg.children())[9:18])
        for param in self.parameters():
            param.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        pred = (pred - self.mean) / self.std
        target = (target - self.mean) / self.std
        pred_f1 = self.slice1(pred)
        pred_f2 = self.slice2(pred_f1)
        pred_f3 = self.slice3(pred_f2)
        with torch.no_grad():
            target_f1 = self.slice1(target)
            target_f2 = self.slice2(target_f1)
            target_f3 = self.slice3(target_f2)
        return F.l1_loss(pred_f1, target_f1) + F.l1_loss(pred_f2, target_f2) + F.l1_loss(pred_f3, target_f3)


class FFTLoss(nn.Module):
    def forward(self, pred, target):
        pred_fft = torch.fft.rfft2(pred)
        target_fft = torch.fft.rfft2(target)
        return F.l1_loss(pred_fft.real, target_fft.real) + F.l1_loss(pred_fft.imag, target_fft.imag)


class GradientLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1))
        self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1))

    def forward(self, pred, target):
        pred_gx = F.conv2d(pred, self.sobel_x, padding=1, groups=3)
        pred_gy = F.conv2d(pred, self.sobel_y, padding=1, groups=3)
        target_gx = F.conv2d(target, self.sobel_x, padding=1, groups=3)
        target_gy = F.conv2d(target, self.sobel_y, padding=1, groups=3)
        return F.l1_loss(pred_gx, target_gx) + F.l1_loss(pred_gy, target_gy)


class WaveletHFLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dwt = DWT()

    def forward(self, pred, target):
        _, pred_lh, pred_hl, pred_hh = self.dwt(pred)
        _, target_lh, target_hl, target_hh = self.dwt(target)
        return F.l1_loss(pred_lh, target_lh) + F.l1_loss(pred_hl, target_hl) + F.l1_loss(pred_hh, target_hh)


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.vgg = VGGPerceptualLoss()
        self.fft = FFTLoss()
        self.gradient = GradientLoss()
        self.wavelet_hf = WaveletHFLoss()
        self.w_l1 = 1.0
        self.w_vgg = 0.1
        self.w_fft = 0.05
        self.w_gradient = 0.1
        self.w_wavelet = 0.02

    def forward(self, pred, target):
        l1_loss = self.l1(pred, target)
        vgg_loss = self.vgg(pred, target)
        fft_loss = self.fft(pred, target)
        gradient_loss = self.gradient(pred, target)
        wavelet_loss = self.wavelet_hf(pred, target)
        total_loss = (
            self.w_l1 * l1_loss
            + self.w_vgg * vgg_loss
            + self.w_fft * fft_loss
            + self.w_gradient * gradient_loss
            + self.w_wavelet * wavelet_loss
        )
        return total_loss, {
            "l1": float(l1_loss.detach().cpu().item()),
            "vgg": float(vgg_loss.detach().cpu().item()),
            "fft": float(fft_loss.detach().cpu().item()),
            "gradient": float(gradient_loss.detach().cpu().item()),
            "wavelet": float(wavelet_loss.detach().cpu().item()),
        }


class RealBlurJ(Dataset):
    '''RealBlur-J dataset loader: root_dir/{train|test}/{blur|gt}/*.png'''

    def __init__(self, root_dir, split="train", patch_size=256):
        self.root_dir = root_dir
        self.split = split
        self.patch_size = patch_size
        blur_dir = os.path.join(root_dir, split, "blur")
        gt_dir = os.path.join(root_dir, split, "gt")
        self.blur_images = []
        self.sharp_images = []
        blur_files = []
        if os.path.exists(gt_dir) and os.path.exists(blur_dir):
            for ext in ("png", "jpg", "jpeg", "PNG", "JPG", "JPEG"):
                blur_files.extend(glob(os.path.join(blur_dir, f"*.{ext}")))
            # De-dupe: case-insensitive filesystems (Windows) match "*.png"
            # and "*.PNG" twice against the same files. Harmless on Kaggle's
            # Linux (case-sensitive) -- verified this doubles the real count
            # (3758->7516, 980->1960) without it, on this machine.
            blur_files = sorted(set(blur_files))
        missing = 0
        for blur_path in blur_files:
            filename = os.path.basename(blur_path)
            gt_path = os.path.join(gt_dir, filename)
            if os.path.exists(gt_path):
                self.blur_images.append(blur_path)
                self.sharp_images.append(gt_path)
            else:
                missing += 1
        print(f"Found {len(self.blur_images)} {split} image pairs in RealBlur-J (missing: {missing})")
        if len(self.blur_images) == 0:
            print("[DEBUG] RealBlur-J expected folders:")
            print("        blur_dir:", blur_dir)
            print("        gt_dir  :", gt_dir)
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.blur_images)

    def __getitem__(self, idx):
        blur = Image.open(self.blur_images[idx]).convert("RGB")
        sharp = Image.open(self.sharp_images[idx]).convert("RGB")
        blur = self.to_tensor(blur)
        sharp = self.to_tensor(sharp)
        if self.split == "train" and self.patch_size:
            _, h, w = blur.shape
            if h >= self.patch_size and w >= self.patch_size:
                top = np.random.randint(0, h - self.patch_size + 1)
                left = np.random.randint(0, w - self.patch_size + 1)
                blur = blur[:, top : top + self.patch_size, left : left + self.patch_size]
                sharp = sharp[:, top : top + self.patch_size, left : left + self.patch_size]
            if np.random.random() > 0.5:
                blur = torch.flip(blur, [2])
                sharp = torch.flip(sharp, [2])
            if np.random.random() > 0.5:
                blur = torch.flip(blur, [1])
                sharp = torch.flip(sharp, [1])
        return blur, sharp

    def relpaths(self):
        out = []
        for b in self.blur_images:
            out.append(os.path.join(self.split, "blur", os.path.basename(b)).replace("\\", "/"))
        return out

## Metrics, visualization, and training-curve plotting

In [ ]:
def calculate_psnr(pred, target):
    mse = F.mse_loss(pred, target)
    if float(mse.detach().cpu().item()) == 0.0:
        return float("inf")
    return 10 * torch.log10(1.0 / mse)


def calculate_ssim(pred, target, window_size=11):
    C1 = 0.01**2
    C2 = 0.03**2
    sigma = 1.5
    gauss = torch.exp(-torch.arange(window_size, device=pred.device).float().sub(window_size // 2).pow(2) / (2 * sigma**2))
    gauss = gauss / gauss.sum()
    window = gauss.unsqueeze(0) * gauss.unsqueeze(1)
    window = window.unsqueeze(0).unsqueeze(0).expand(3, 1, window_size, window_size)
    mu1 = F.conv2d(pred, window, padding=window_size // 2, groups=3)
    mu2 = F.conv2d(target, window, padding=window_size // 2, groups=3)
    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2
    sigma1_sq = F.conv2d(pred * pred, window, padding=window_size // 2, groups=3) - mu1_sq
    sigma2_sq = F.conv2d(target * target, window, padding=window_size // 2, groups=3) - mu2_sq
    sigma12 = F.conv2d(pred * target, window, padding=window_size // 2, groups=3) - mu1_mu2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean()


# Reference points verified from the actual executed RealBlur-J full-model
# notebook (notebookfb28f3c539 - Copy.ipynb) and from the local GoPro
# spatial-only ablation run (run_20260915_200933), not from memory:
FULL_MODEL_VAL_PSNR_BY_EPOCH = {
    1: 27.30, 10: 28.85, 20: 29.44, 30: 29.70, 40: 29.86, 50: 29.98, 60: 30.07, 70: 30.08,
}
FULL_MODEL_BEST_VAL_PSNR = 30.08
FULL_MODEL_BEST_VAL_SSIM = 0.8877
FULL_MODEL_TEST_PSNR = 29.57
FULL_MODEL_TEST_SSIM = 0.8839

# Spatial-only RealBlur-J, completed Kaggle run (values from its executed notebook output + history JSON):
SPATIAL_ONLY_VAL_PSNR_BY_EPOCH = {1: 27.37, 10: 29.01, 20: 29.50, 30: 29.58, 40: 29.73, 50: 29.92, 60: 30.04, 70: 30.04}
SPATIAL_ONLY_TEST_PSNR = 29.53
SPATIAL_ONLY_TEST_SSIM = 0.8819

# VAL results of the interrupted fixed-fusion run: epochs 1/10/20/30 from the saved console log, epoch 40
# read from a photo of the console, epoch 50 read from the checkpoint file itself.
PRE_RESUME_VAL_EPOCHS = [1, 10, 20, 30, 40, 50]
PRE_RESUME_VAL_PSNRS = [27.60, 29.00, 29.57, 29.68, 29.90, 30.0121]
PRE_RESUME_VAL_SSIMS = [0.8083, 0.8595, 0.8731, 0.8760, 0.8839, 0.8853]


def visualize_results(model, test_loader, device, save_dir, num_samples=10):
    print("\n=== Generating Visual Comparisons (TEST) ===")
    model.eval()
    results_dir = os.path.join(save_dir, "results_ablation_nogate_realblurj")
    os.makedirs(results_dir, exist_ok=True)
    total_samples = len(test_loader.dataset)
    selected_indices = random.sample(range(total_samples), min(num_samples, total_samples))
    selected_indices.sort()
    print(f"Randomly selected {len(selected_indices)} test images for visualization...")
    with torch.no_grad():
        for idx, sample_idx in enumerate(selected_indices):
            blur, sharp = test_loader.dataset[sample_idx]
            blur = blur.unsqueeze(0).to(device)
            sharp = sharp.unsqueeze(0).to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                pred = model(blur)
            pred = torch.clamp(pred, 0, 1)
            psnr = float(calculate_psnr(pred, sharp).detach().cpu().item())
            ssim = float(calculate_ssim(pred, sharp).detach().cpu().item())
            blur_np = blur.squeeze(0).cpu().numpy().transpose(1, 2, 0)
            pred_np = pred.squeeze(0).cpu().numpy().transpose(1, 2, 0)
            sharp_np = sharp.squeeze(0).cpu().numpy().transpose(1, 2, 0)
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            axes[0].imshow(blur_np); axes[0].set_title("Blur Input"); axes[0].axis("off")
            axes[1].imshow(pred_np); axes[1].set_title(f"Deblurred (Fixed-Fusion)\nPSNR: {psnr:.2f} dB | SSIM: {ssim:.4f}"); axes[1].axis("off")
            axes[2].imshow(sharp_np); axes[2].set_title("Ground Truth"); axes[2].axis("off")
            plt.tight_layout()
            plt.savefig(os.path.join(results_dir, f"sample_{idx}_comparison.png"), dpi=150, bbox_inches="tight")
            plt.close(fig)
    print(f"Saved {len(selected_indices)} comparisons to: {results_dir}")
    return results_dir


def plot_training_curves(save_dir, history):
    curve_path = os.path.join(save_dir, "ablation_nogate_realblurj_training_curves.png")
    fig, ax1 = plt.subplots(figsize=(10, 5))
    if history["train_losses"]:
        ax1.plot(history["loss_epochs"], history["train_losses"], color="tab:red", alpha=0.6, label="Fixed-Fusion Train Loss (resumed epochs)")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Train Loss", color="tab:red"); ax1.tick_params(axis="y", labelcolor="tab:red")
    ax2 = ax1.twinx()
    ax2.plot(history["val_epochs"], history["val_psnrs"], color="tab:purple", marker="d", label="Fixed-Fusion VAL PSNR")
    full_epochs = sorted(FULL_MODEL_VAL_PSNR_BY_EPOCH.keys())
    ax2.plot(full_epochs, [FULL_MODEL_VAL_PSNR_BY_EPOCH[e] for e in full_epochs], color="tab:green", marker="^", linestyle="--", label="Full Model VAL PSNR (reference)")
    sp_epochs = sorted(SPATIAL_ONLY_VAL_PSNR_BY_EPOCH.keys())
    ax2.plot(sp_epochs, [SPATIAL_ONLY_VAL_PSNR_BY_EPOCH[e] for e in sp_epochs], color="tab:blue", marker="s", linestyle=":", label="Spatial-Only VAL PSNR (reference)")
    ax2.axvline(history["resume_epoch"], color="gray", linestyle="-.", alpha=0.7)
    ax2.set_ylabel("VAL PSNR (dB)", color="tab:purple"); ax2.tick_params(axis="y", labelcolor="tab:purple")
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc="lower right")
    plt.title(f"Ablation: Fixed-Fusion (resumed at epoch {history['resume_epoch']}) vs. Spatial-Only vs. Full Model -- RealBlur-J")
    plt.tight_layout()
    plt.savefig(curve_path, dpi=150)
    plt.close(fig)
    print(f"Training curves saved to: {curve_path}")
    return curve_path

## Environment-aware config (auto-detects Kaggle vs. local -- no manual path editing needed either way)

In [ ]:
# This resume notebook is local-only: the epoch-50 checkpoint lives on this machine.
DEFAULT_DATA_ROOT = r"C:\Users\user\Desktop\APP\APP\APP\under12hrs_code\FINAL_CONFERENCE\REAL_BLUR_J_CONFERENCE\rbj\RealBlur"
DEFAULT_SAVE_ROOT = r"C:\Users\user\Desktop\APP\APP\APP\under12hrs_code\ABLATION\runs"

RESUME_CKPT = r"C:\Users\user\Desktop\APP\APP\APP\under12hrs_code\ABLATION\runs\run_nogate_realblurj_20260923_164622\best_model_ablation_nogate_realblurj.pth"
RESUME_EXPECTED_EPOCH = 50

print("Data root:", DEFAULT_DATA_ROOT)
print("Resuming from:", RESUME_CKPT)

_RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
RUN_SAVE_DIR = os.path.join(DEFAULT_SAVE_ROOT, f"run_nogate_realblurj_RESUME_{_RUN_STAMP}")
print("This run's save dir:", RUN_SAVE_DIR)

## Resume training (epochs 51-70)

In [ ]:
import math
import shutil
import sys


def _keep_system_awake(enable):
    # Windows only. Asks Windows not to idle-sleep while this kernel is running. It is a per-process
    # request that ends when disabled or when the kernel exits; it changes no Windows setting.
    # A manual Sleep, or a lid-close whose action is "Sleep", still sleeps the PC (see the intro cell).
    if sys.platform != "win32":
        return False
    import ctypes
    ES_CONTINUOUS = 0x80000000
    ES_SYSTEM_REQUIRED = 0x00000001
    flags = ES_CONTINUOUS | (ES_SYSTEM_REQUIRED if enable else 0)
    return bool(ctypes.windll.kernel32.SetThreadExecutionState(flags))


def _closed_form_cosine_lr(epoch_index, base_lr, eta_min, t_max):
    return eta_min + (base_lr - eta_min) * (1 + math.cos(math.pi * epoch_index / t_max)) / 2


def build_optimizer_and_scheduler_at_epoch(model, config, start_epoch):
    # Fresh AdamW (moment estimates are not in a weights-only checkpoint), with the cosine schedule
    # advanced to exactly where it was after `start_epoch` completed epochs.
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"], eta_min=config["min_lr"])
    for _ in range(start_epoch):
        optimizer.step()  # no gradients exist yet, so this changes nothing; it keeps the scheduler call order valid
        scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]
    lr_expected = _closed_form_cosine_lr(start_epoch, config["lr"], config["min_lr"], config["epochs"])
    if abs(lr_now - lr_expected) > 0.01 * lr_expected:
        raise RuntimeError(f"Scheduler restore failed: lr={lr_now:.4e}, expected {lr_expected:.4e}")
    print(f"Cosine schedule restored to epoch {start_epoch}: lr for the next epoch = {lr_now:.3e} (closed form {lr_expected:.3e})")
    return optimizer, scheduler


def load_resume_weights(model, ckpt_path, expected_epoch):
    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f"Resume checkpoint not found: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if int(ckpt["epoch"]) != int(expected_epoch):
        raise RuntimeError(f"Checkpoint holds epoch {ckpt['epoch']}, expected {expected_epoch}")
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded weights from epoch {ckpt['epoch']} (stored VAL PSNR {ckpt['val_psnr']:.4f} dB, SSIM {ckpt['val_ssim']:.4f})")
    return ckpt


def assert_same_split(old_proof_path, new_proof_path):
    old = json.load(open(old_proof_path, encoding="utf-8"))
    new = json.load(open(new_proof_path, encoding="utf-8"))
    for key in ("train", "val", "test"):
        if old["sha256"][key] != new["sha256"][key]:
            raise RuntimeError(f"Split mismatch on '{key}': this run would NOT use the same data as the interrupted run.")
    print("Split check passed: train/val/test SHA256 hashes are identical to the interrupted run's.")


def verify_resumed_weights(model, val_loader, device, stored_psnr, tolerance_db=0.05):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc="Sanity check: VAL with resumed weights"):
            blur = blur.to(device)
            sharp = sharp.to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                pred = model(blur)
            pred = torch.clamp(pred, 0, 1)
            total += float(calculate_psnr(pred, sharp).detach().cpu().item())
    measured = total / len(val_loader)
    print(f"Sanity check: VAL PSNR with loaded weights = {measured:.4f} dB (stored in checkpoint: {stored_psnr:.4f} dB)")
    if abs(measured - stored_psnr) > tolerance_db:
        raise RuntimeError("Loaded weights do not reproduce the stored VAL PSNR -- refusing to spend hours training from a bad restore.")
    return measured


def save_history(save_dir, history):
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, "ablation_nogate_realblurj_history.json")
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    os.replace(tmp, path)
    return path


def train_ablation_nogate_realblurj_resume():
    config = {
        "data_root": DEFAULT_DATA_ROOT,
        "batch_size": 1,
        "grad_accum_steps": 2,
        "patch_size": 192,
        "epochs": 70,
        "lr": 2e-4,
        "min_lr": 1e-7,
        "num_workers": 0,
        "val_num_workers": 0,
        "test_num_workers": 0,
        "base_channels": 48,
        "num_blocks": (4, 6, 6, 4),
        "save_dir": RUN_SAVE_DIR,
        "val_ratio": 0.1,
        "split_seed": 42,
        "cuda_empty_cache_interval": 50,
    }
    start_epoch = int(RESUME_EXPECTED_EPOCH)

    seed_everything(int(config["split_seed"]))
    config["data_root"] = resolve_realblur_root(config["data_root"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if device.type != "cuda":
        raise RuntimeError("No CUDA GPU detected -- refusing to start; this would take days on CPU.")
    torch.backends.cudnn.benchmark = False

    model = WaveFusionNetFixedFusion(base_channels=config["base_channels"], num_blocks=config["num_blocks"])
    ckpt = load_resume_weights(model, RESUME_CKPT, start_epoch)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {total_params:,} (expected 9,392,211)")
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    model = model.to(device)

    criterion = CombinedLoss().to(device)
    optimizer, scheduler = build_optimizer_and_scheduler_at_epoch(model, config, start_epoch)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    full_train_aug = RealBlurJ(config["data_root"], split="train", patch_size=config["patch_size"])
    full_train_noaug = RealBlurJ(config["data_root"], split="train", patch_size=None)
    test_dataset = RealBlurJ(config["data_root"], split="test", patch_size=None)
    n_total = len(full_train_aug)
    if n_total == 0:
        raise RuntimeError("No training samples found. Check data_root.")

    rng = np.random.default_rng(int(config["split_seed"]))
    indices = np.arange(n_total)
    rng.shuffle(indices)
    n_val = max(1, int(n_total * float(config["val_ratio"])))
    val_indices = indices[:n_val].tolist()
    train_indices = indices[n_val:].tolist()

    train_dataset = Subset(full_train_aug, train_indices)
    val_dataset = Subset(full_train_noaug, val_indices)
    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True,
                               num_workers=config["num_workers"], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                             num_workers=int(config["val_num_workers"]), pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                              num_workers=int(config["test_num_workers"]), pin_memory=True)

    train_rel = [full_train_aug.relpaths()[i] for i in train_indices]
    val_rel = [full_train_noaug.relpaths()[i] for i in val_indices]
    test_rel = test_dataset.relpaths()
    proof_path = export_split_proof(config["save_dir"], train_rel, val_rel, test_rel, config)
    print(f"Split sizes: train={len(train_indices)} val={len(val_indices)} test={len(test_dataset)}")
    old_proof = os.path.join(os.path.dirname(RESUME_CKPT), "split_proof_ablation_nogate_realblurj.json")
    assert_same_split(old_proof, proof_path)
    verify_resumed_weights(model, val_loader, device, float(ckpt["val_psnr"]))

    best_model_path = os.path.join(config["save_dir"], "best_model_ablation_nogate_realblurj.pth")
    shutil.copy2(RESUME_CKPT, best_model_path)  # so the best-VAL file always exists, even if epochs 60/70 do not beat epoch 50
    best_val_psnr = float(ckpt["val_psnr"])
    best_val_ssim = float(ckpt["val_ssim"])

    history = {
        "run_type": "RESUMED from epoch-50 weights (fresh AdamW state; cosine schedule restored to epoch 50)",
        "resume_epoch": start_epoch,
        "resume_checkpoint": RESUME_CKPT,
        "val_epochs": list(PRE_RESUME_VAL_EPOCHS),
        "val_psnrs": list(PRE_RESUME_VAL_PSNRS),
        "val_ssims": list(PRE_RESUME_VAL_SSIMS),
        "pre_resume_values_source": "epochs 1/10/20/30: console log; epoch 40: photo of console; epoch 50: read from the checkpoint file",
        "loss_epochs": [],
        "train_losses": [],
        "last_completed_epoch": start_epoch,
        "full_model_reference": FULL_MODEL_VAL_PSNR_BY_EPOCH,
        "spatial_only_reference": SPATIAL_ONLY_VAL_PSNR_BY_EPOCH,
    }
    save_history(config["save_dir"], history)

    # New seed offset so the resumed epochs do not replay epoch 1's sample order / crops.
    # (The train/val split above uses its own generator, so it is unaffected.)
    seed_everything(int(config["split_seed"]) + start_epoch)

    _keep_system_awake(True)
    print("\nKeep-awake requested. Reminder: do NOT press Sleep, and set lid-close to 'Do nothing' while this runs.")
    try:
        print("\n=== Resuming Ablation Training (Fixed-Fusion) ===")
        for epoch in range(start_epoch, int(config["epochs"])):
            model.train()
            epoch_loss = 0.0
            accum_steps = max(1, int(config.get("grad_accum_steps", 1)))
            optimizer.zero_grad(set_to_none=True)
            pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config['epochs']}")
            for step, (blur, sharp) in enumerate(pbar, start=1):
                blur = blur.to(device, non_blocking=True)
                sharp = sharp.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    pred = model(blur)
                    loss, components = criterion(pred, sharp)
                loss_value = float(loss.detach().cpu().item())
                loss = loss / accum_steps
                scaler.scale(loss).backward()
                if step % accum_steps == 0 or step == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                epoch_loss += loss_value
                pbar.set_postfix({"loss": f"{loss_value:.4f}"})
                del pred, loss, blur, sharp
                if device.type == "cuda" and step % int(config["cuda_empty_cache_interval"]) == 0:
                    torch.cuda.empty_cache()
                    gc.collect()

            scheduler.step()
            epoch_loss /= max(1, len(train_loader))
            history["loss_epochs"].append(epoch + 1)
            history["train_losses"].append(epoch_loss)
            history["last_completed_epoch"] = epoch + 1
            save_history(config["save_dir"], history)

            is_final_epoch = (epoch + 1) == int(config["epochs"])
            if (epoch + 1) % 10 == 0 or is_final_epoch:
                model.eval()
                val_psnr = 0.0
                val_ssim = 0.0
                with torch.no_grad():
                    for blur, sharp in tqdm(val_loader, desc="Validating (VAL)"):
                        blur = blur.to(device)
                        sharp = sharp.to(device)
                        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                            pred = model(blur)
                        pred = torch.clamp(pred, 0, 1)
                        val_psnr += float(calculate_psnr(pred, sharp).detach().cpu().item())
                        val_ssim += float(calculate_ssim(pred, sharp).detach().cpu().item())
                val_psnr /= len(val_loader)
                val_ssim /= len(val_loader)
                history["val_epochs"].append(epoch + 1)
                history["val_psnrs"].append(val_psnr)
                history["val_ssims"].append(val_ssim)

                print(f"\nEpoch {epoch + 1}: Loss={epoch_loss:.4f}, VAL PSNR={val_psnr:.2f}dB, VAL SSIM={val_ssim:.4f}")
                ref_full = FULL_MODEL_VAL_PSNR_BY_EPOCH.get(epoch + 1)
                ref_spatial = SPATIAL_ONLY_VAL_PSNR_BY_EPOCH.get(epoch + 1)
                if ref_full is not None:
                    print(f"  >>> vs. full model at epoch {epoch + 1}: fixed-fusion={val_psnr:.2f}dB, full model={ref_full:.2f}dB (diff {val_psnr - ref_full:+.2f}dB)")
                if ref_spatial is not None:
                    print(f"  >>> vs. spatial-only at epoch {epoch + 1}: fixed-fusion={val_psnr:.2f}dB, spatial-only={ref_spatial:.2f}dB (diff {val_psnr - ref_spatial:+.2f}dB)")

                if val_psnr > best_val_psnr:
                    best_val_psnr = val_psnr
                    best_val_ssim = val_ssim
                    state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
                    torch.save({"epoch": epoch + 1, "model_state_dict": state_dict, "val_psnr": val_psnr, "val_ssim": val_ssim}, best_model_path)
                    print(f"  *** New best fixed-fusion model saved! VAL PSNR: {val_psnr:.2f}dB ***")
                save_history(config["save_dir"], history)
                if device.type == "cuda":
                    torch.cuda.empty_cache()
                    gc.collect()

        print("\n=== Resumed Training Complete ===")
        print(f"Best VAL PSNR: {best_val_psnr:.2f}dB | Best VAL SSIM: {best_val_ssim:.4f}")

        print("\n=== Final Test Evaluation (TEST) ===")
        checkpoint = torch.load(best_model_path, map_location=device)
        state = checkpoint.get("model_state_dict", checkpoint)
        if hasattr(model, "module"):
            model.module.load_state_dict(state)
        else:
            model.load_state_dict(state)
        print(f"Loaded best-VAL fixed-fusion model from epoch {checkpoint.get('epoch', 'N/A')}")

        model.eval()
        test_psnr = 0.0
        test_ssim = 0.0
        with torch.no_grad():
            for blur, sharp in tqdm(test_loader, desc="Testing (TEST)"):
                blur = blur.to(device)
                sharp = sharp.to(device)
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    pred = model(blur)
                pred = torch.clamp(pred, 0, 1)
                test_psnr += float(calculate_psnr(pred, sharp).detach().cpu().item())
                test_ssim += float(calculate_ssim(pred, sharp).detach().cpu().item())
        test_psnr /= len(test_loader)
        test_ssim /= len(test_loader)
        history["best_checkpoint_epoch"] = int(checkpoint.get("epoch", -1))
        history["test_psnr"] = test_psnr
        history["test_ssim"] = test_ssim
        save_history(config["save_dir"], history)

        print("\n" + "=" * 80)
        print("FINAL THREE-WAY COMPARISON (RealBlur-J, identical seed/split/protocol)")
        print("=" * 80)
        print(f"Full model (learned gate):            TEST PSNR={FULL_MODEL_TEST_PSNR:.2f}dB, TEST SSIM={FULL_MODEL_TEST_SSIM:.4f}, params=9,484,659")
        print(f"Fixed-fusion (RESUMED at epoch {start_epoch}):  TEST PSNR={test_psnr:.2f}dB, TEST SSIM={test_ssim:.4f}, params={total_params:,}")
        print(f"Spatial-only (Kaggle):                TEST PSNR={SPATIAL_ONLY_TEST_PSNR:.2f}dB, TEST SSIM={SPATIAL_ONLY_TEST_SSIM:.4f}, params=5,305,635")
        print(f"\nFixed-fusion vs. full model:   {test_psnr - FULL_MODEL_TEST_PSNR:+.2f}dB PSNR, {test_ssim - FULL_MODEL_TEST_SSIM:+.4f} SSIM")
        print(f"Fixed-fusion vs. spatial-only: {test_psnr - SPATIAL_ONLY_TEST_PSNR:+.2f}dB PSNR, {test_ssim - SPATIAL_ONLY_TEST_SSIM:+.4f} SSIM")
        print("Report this run as: fixed-fusion, resumed at epoch 50 with re-initialised optimiser state.")
        print("=" * 80)

        plot_training_curves(config["save_dir"], history)
        visualize_results(model, test_loader, device, config["save_dir"], num_samples=10)

        print("\n" + "=" * 80)
        print("ALL OUTPUTS GENERATED THIS RUN (all under):", config["save_dir"])
        print("=" * 80)
        print("1. Best checkpoint:          best_model_ablation_nogate_realblurj.pth")
        print("2. Split proof (SHA256):     split_proof_ablation_nogate_realblurj.json")
        print("3. Training curves plot:     ablation_nogate_realblurj_training_curves.png")
        print("4. History + TEST numbers:   ablation_nogate_realblurj_history.json")
        print("5. Sample comparisons:       results_ablation_nogate_realblurj/sample_*_comparison.png")
        print("=" * 80)
        return test_psnr, test_ssim
    finally:
        _keep_system_awake(False)

## Run it

In [ ]:
test_psnr, test_ssim = train_ablation_nogate_realblurj_resume()